<span style="color:gray;">Machine learning>> Redes neuronales >>Perceptrón Multicapa>> Regresión </span>
<h1 style="color:blue;margin-top:2px;">Perceptrón multicapa para regresión</h1>
<h2>Caso de uso: predicción de precios vivienda</h2>
El set de datos SaratogaHouses del paquete mosaicData de R contiene información sobre el precio de 1728 viviendas situadas en Saratoga County, New York, USA en el año 2006. Además del precio, incluye 15 variables adicionales:
<ul>
    <li><b style="display:inline-block;width:90px;">price</b>: precio de la vivienda.</li>
    <li><b style="display:inline-block;width:90px;">lotSize</b>: metros cuadrados de la vivienda.</li>
    <li><b style="display:inline-block;width:90px;">age</b>: antigüedad de la vivienda.</li>
    <li><b style="display:inline-block;width:90px;">landValue</b>: valor del terreno.</li>
    <li><b style="display:inline-block;width:90px;">livingArea</b>: metros cuadrados habitables.</li>
    <li><b style="display:inline-block;width:90px;">pctCollege</b>: porcentaje del vecindario con título universitario.</li>
    <li><b style="display:inline-block;width:90px;">bedrooms</b>: número de dormitorios.</li>
    <li><b style="display:inline-block;width:90px;">firplaces</b>: número de chimeneas.</li>
    <li><b style="display:inline-block;width:90px;">bathrooms</b>: número de cuartos de baño (el valor 0.5 hace referencia a cuartos de baño sin ducha).</li>
    <li><b style="display:inline-block;width:90px;">rooms</b>: número de habitaciones.</li>
    <li><b style="display:inline-block;width:90px;">heating</b>: tipo de calefacción.</li>
    <li><b style="display:inline-block;width:90px;">fuel</b>: tipo de alimentación de la calefacción (gas, electricidad o diesel).</li>
    <li><b style="display:inline-block;width:90px;">sewer</b>: tipo de desagüe.</li>
    <li><b style="display:inline-block;width:90px;">waterfront</b>: si la vivienda tiene vistas al lago.</li>
    <li><b style="margin-right:15px;">newConstruction</b>: si la vivienda es de nueva construcción.</li>
    <li><b style="display:inline-block;width:90px;">centralAir</b>: si la vivienda tiene aire acondicionado.</li>
</ul>
El objetivo es obtener un modelo capaz de predecir el precio del alquiler.

<h2>Librerías</h2>

In [ ]:
# Tratamiento de datos
# ==============================================================================
import numpy as np
import pandas as pd

# Gráficos
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('fivethirtyeight')

# Modelado
# ==============================================================================
from sklearn.neural_network import MLPRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
from sklearn import set_config
import multiprocessing

# Configuración warnings
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

<h2>Datos</h2>
Puede obtener el repositorio en este enlace:<br/>
<a href="https://r-data.pmagunia.com/dataset/r-dataset-package-mosaicdata-saratogahouses">SaratogaHouses.csv</a>

In [ ]:
datos = pd.read_csv("SaratogaHouses.csv")

# Se renombran las columnas para que sean más descriptivas
datos.columns = ["precio", "metros_totales", "antiguedad", "precio_terreno",
                 "metros_habitables", "universitarios", "dormitorios", 
                 "chimenea", "banyos", "habitaciones", "calefaccion",
                 "consumo_calefacion", "desague", "vistas_lago",
                 "nueva_construccion", "aire_acondicionado"]

In [ ]:
datos.head()

<h2>Análisis exploratorio</h2>
Antes de entrenar un modelo predictivo, o incluso antes de realizar cualquier cálculo con un nuevo conjunto de datos, es muy importante realizar una exploración descriptiva de los mismos. Este proceso permite entender mejor qué información contiene cada variable, así como detectar posibles errores. Algunos ejemplos frecuentes son:
<ul>
    <li>Que una columna se haya almacenado con el tipo incorrecto: una variable numérica está siendo reconocida como texto o viceversa.</li>
    <li>Que una variable contenga valores que no tienen sentido: por ejemplo, para indicar que no se dispone del precio de una vivienda se introduce el valor 0 o un espacio en blanco.</li>
    <li>Que en una variable de tipo numérico se haya introducido una palabra en lugar de un número.</li>
</ul>
Además, este análisis inicial puede dar pistas sobre qué variables son adecuadas como predictores en un modelo.

In [ ]:
# Tipo de cada columna
# En pandas, el tipo "object" hace referencia a strings
datos.info()

Todas las columnas tienen el tipo de dato correcto.

In [ ]:
# Número de datos ausentes por variable
datos.isna().sum().sort_values()

Todas las columnas están completas, no hay valores ausentes.

In [ ]:
# Distribución variable respuesta
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 3))
sns.histplot(data=datos, x='precio', kde=True,ax=ax)
ax.set_title("Distribución Precio")
ax.set_xlabel('precio');

Los modelos de redes neuronales son de tipo <span style="color:red">no paramétrico</span>, no asumen ningún tipo de distribución de la variable respuesta, por lo tanto, no es necesario que esta siga ninguna distribución concreta (normal, gamma...). Aun así, siempre es recomendable hacer un estudio mínimo, ya que, a fin de cuentas, es lo que interesa predecir. En este caso, la variable precio tiene una distribución asimétrica con una cola positiva debido a que, unas pocas viviendas, tienen un precio muy superior a la media.

In [ ]:
# Gráfico de distribución para cada variable numérica
# ==============================================================================
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(12, 7))
axes = axes.flat
columnas_numeric = datos.select_dtypes(include=['float64', 'int']).columns
columnas_numeric = columnas_numeric.drop('precio')

for i, colum in enumerate(columnas_numeric):
    sns.histplot(
        data    = datos,
        x       = colum,
        stat    = "count",
        kde     = True,
        color   = (list(plt.rcParams['axes.prop_cycle'])*2)[i]["color"],
        line_kws= {'linewidth': 2},
        alpha   = 0.3,
        ax      = axes[i]
    )
    axes[i].set_title(colum, fontsize = 7, fontweight = "bold")
    axes[i].tick_params(labelsize = 6)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")
    
    
fig.tight_layout()
plt.subplots_adjust(top = 0.9)
fig.suptitle('Distribución variables numéricas', fontsize = 10, fontweight = "bold");

La variable chimenea, apenas toma unos pocos valores y la gran mayoría de observaciones pertenecen a solo dos de ellos. En casos como este, suele ser conveniente agrupar los valores poco representados en categoria otros, o en este caso tiene=1 o no=0.

In [ ]:
datos['chimenea']=datos['chimenea'].replace([2, 3, 4], 1)

In [ ]:
datos['chimenea'].unique()

In [ ]:
# Gráfico para cada variable cualitativa
# ==============================================================================
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 5))
axes = axes.flat
columnas_object = datos.select_dtypes(include=['object']).columns

for i, colum in enumerate(columnas_object):
    datos[colum].value_counts().plot.barh(ax = axes[i])
    axes[i].set_title(colum, fontsize = 14)
    axes[i].set_xlabel("")
    
fig.tight_layout()

In [ ]:
print("vistas_lago\n",datos.vistas_lago.value_counts())
print("\n\nnueva_construccion\n",datos.nueva_construccion.value_counts())
print("\n\ndesague\n",datos.desague.value_counts())

In [ ]:
datos['aire_acondicionado']=datos['aire_acondicionado'].replace("Yes",1)
datos['aire_acondicionado']=datos['aire_acondicionado'].replace("No", 0)

Por otro lado que la propiedad sea nueva o no, no suele ser tan determinante y al estar muy desbalanceados los valores la eliminaremos para no aumentar la complejidad del modelo inecesariamente.

In [ ]:
datos.drop(columns="nueva_construccion",axis=1,inplace=True)

In [ ]:
datos.drop(columns="vistas_lago",axis=1,inplace=True)

In [ ]:
datos.shape

Si alguno de los niveles de una variable cualitativa tiene muy pocas observaciones en comparación a los otros niveles, puede ocurrir que, durante la <b>validación cruzada o bootstrapping</b>, algunas particiones no contengan ninguna observación de dicha clase (varianza cero), lo que puede dar lugar a errores. Para este caso, hay que tener precaución con la variable desague. Se unifican los niveles de none y septic en un nuevo nivel llamado "otro desague".

In [ ]:
datos['desague']=datos['desague'].replace("none", "otro_desague")
datos['desague']=datos['desague'].replace("septic", "otro_desague")

Verificamos si hay columnas numéricas con poca o nula varianza

In [ ]:
num_cols = datos.select_dtypes('number').columns
low_var_cols = []
for col in num_cols:
    scaled = (datos[col] - datos[col].mean()) / datos[col].std()
    variance = scaled.var().round(4)
    if variance == 0 or datos[col].std() == 0:
        low_var_cols.append(col)
    # else:
        # print(col, variance)
datos.drop(columns=low_var_cols, axis=1, inplace=True)
print(f"Removed Columns: {low_var_cols}")
#print(data.shape, len(low_var_cols))

In [ ]:
X=datos.drop(columns="precio",axis=1)

In [ ]:
print("Matriz de Correlación")
plt.rcParams['figure.figsize']=(15,6)
sns.heatmap(X.corr(),cmap='coolwarm',linewidths=.5,fmt=".2f",annot = True);

Aunque metros habitables presenta una correlación alta (>0.7) con el número de baños y el número de habitaciones las tres son características muy importantes al momento de determinar el precio de una propiedad por lo que las conservaremos.<br/>
Aunque ya revisamos que ninguna columna categórica presenta una cardinalidad alta (>5 categorias) podemos verificar con la siguiente prueba.

In [ ]:
cat_cols = datos.select_dtypes('object').columns
k = 0
cardinality_cols = []
for col in cat_cols:
    value_counts = datos[col].value_counts(normalize=True).round(2)
    if len(value_counts) > 5 or len(value_counts) < 2:
        cardinality_cols.append(col)
    else:
        print("======================================================")
        print(col)
        print(value_counts)
print(f"Removed Columns: {cardinality_cols}")
datos.drop(columns=cardinality_cols, axis=1, inplace=True)

In [ ]:
X.drop(columns=["calefaccion","consumo_calefacion","desague"],axis=1,inplace=True)

In [ ]:
X.drop(columns=["metros_habitables","habitaciones","dormitorios","banyos"],axis=1,inplace=True)

In [ ]:
X.dtypes

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del datos ['universitarios']
del X ['universitarios']
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

<h2>División train y test</h2>
Con el objetivo de poder estimar el error que comete el modelo al predecir nuevas observaciones, se dividen los datos en dos grupos, uno de entrenamiento y otro de test (80%, 20%).

In [ ]:
# Reparto de datos en train y test
# ==============================================================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                        datos.drop('precio', axis = 'columns'),
                                        datos['precio'],
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

Tras realizar el reparto, se verifica que los dos grupos son similares.

In [ ]:
print("Partición de entrenamento")
print("-----------------------")
display(y_train.describe())
display(X_train.describe())
display(X_train.describe(include = 'object'))
print(" ")

print("Partición de test")
print("-----------------------")
display(y_test.describe())
display(X_test.describe())
display(X_test.describe(include = 'object'))

<h2>Preprocesado</h2>
Los modelos de redes neuronales requieren SIEMPRE como mínimo de dos tipos de preprocesado: binarización (One hot ecoding) de las variables categóricas y estandarización de las variables continuas.

In [ ]:
# Selección de las variables por típo
# ==============================================================================
# Se estandarizan las columnas numéricas y se hace one-hot-encoding de las 
# columnas cualitativas. Para mantener las columnas a las que no se les aplica 
# ninguna transformación se tiene que indicar remainder='passthrough'.

# Identificación de columnas numéricas y categóricas
numeric_cols = X_train.select_dtypes(include=['float64', 'int']).columns.to_list()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.to_list()


# Transformaciones para las variables numéricas
numeric_transformer = Pipeline(
                        steps=[('scaler', StandardScaler())]
                      )

# Transformaciones para las variables categóricas
categorical_transformer = Pipeline(
                            steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))]
                          )

preprocessor = ColumnTransformer(
                    transformers=[
                        ('numeric', numeric_transformer, numeric_cols),
                        ('cat', categorical_transformer, cat_cols)
                    ],
                    remainder='passthrough'
                )

In [ ]:
set_config(display='diagram')
preprocessor

In [ ]:
set_config(display='text')

In [ ]:
# Se aprenden y aplican las transformaciones de preprocesado
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

In [ ]:
# Convertir el output en dataframe y añadir el nombre de las columnas
encoded_cat = preprocessor.named_transformers_['cat']['onehot']\
              .get_feature_names_out(cat_cols)
labels = np.concatenate([numeric_cols, encoded_cat])
datos_train_prep = preprocessor.transform(X_train)
datos_train_prep = pd.DataFrame(datos_train_prep, columns=labels)
datos_train_prep.info()

Si bien realizar el preprocesado de forma separada del entrenamiento es útil para explorar y confirmar que las transformaciones realizadas son las deseadas, en la práctica, es más adecuado asociarlo al propio proceso de entrenamiento. Esto puede hacerse fácilmente en los modelos de scikit-learn con los <span style="color:red">Pipeline</span>.

<h2>Modelado</h2>
<h5>Pipepeline de preprocesado + modelado</h5>

In [ ]:
# Pipeline de preprocedado y modelado
# ==============================================================================

# Identificación de columnas numéricas y categóricas
numeric_cols = X_train.select_dtypes(include=['float64', 'int']).columns.to_list()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.to_list()


# Transformaciones para las variables numéricas
numeric_transformer = Pipeline(
                        steps=[('scaler', StandardScaler())]
                      )

# Transformaciones para las variables categóricas
categorical_transformer = Pipeline(
                            steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))]
                          )

preprocessor = ColumnTransformer(
                    transformers=[
                        ('numeric', numeric_transformer, numeric_cols),
                        ('cat', categorical_transformer, cat_cols)
                    ],
                    remainder='passthrough'
                )

# Se combinan los pasos de preprocesado y el modelo en un mismo pipeline
pipe = Pipeline([('preprocessing', preprocessor),
                 ('modelo', MLPRegressor(solver = 'lbfgs', max_iter= 1000, random_state=123))])

In [ ]:
# Espacio de búsqueda de cada hiperparámetro
param_distributions = {
    'modelo__hidden_layer_sizes': [30,50,70,90, (7,7),(8,8),(9,9)],
    'modelo__alpha': np.logspace(-3, 3, 10),
    'modelo__learning_rate_init': [0.00001,0.0001,0.001, 0.01],
    'modelo__activation':['logistic', 'tanh', 'relu'],
}

# Búsqueda por validación cruzada
grid = RandomizedSearchCV(
        estimator  = pipe,
        param_distributions = param_distributions,
        n_iter     = 50,
        scoring    = 'neg_mean_squared_error',
        n_jobs     = multiprocessing.cpu_count() - 1,
        cv         = 5, 
        verbose    = 0,
        random_state = 123,
        return_train_score = True
       )

grid.fit(X = X_train, y = y_train)

# Resultados del grid
resultados = pd.DataFrame(grid.cv_results_)
resultados.filter(regex = '(param.*|mean_t|std_t)')\
    .drop(columns = 'params')\
    .sort_values('mean_test_score', ascending = False)\
    .head(20)

<h2>Error de test</h2>
Aunque mediante los métodos de validación (Kfold, LeaveOneOut) se consiguen buenas estimaciones del error que tiene un modelo al predecir nuevas observaciones, la mejor forma de evaluar un modelo final es prediciendo un conjunto test, es decir, un conjunto de observaciones que se ha mantenido al margen del proceso de entrenamiento y optimización.

In [ ]:
# Error de test
modelo_final = grid.best_estimator_
predicciones = modelo_final.predict(X = X_test)
rmse = mean_squared_error(
        y_true = y_test,
        y_pred = predicciones,
        squared = False
       )
print('Error de test (rmse): ', rmse)

<h2>Conclusión</h2>
La combinación de hiperparámetros con la que se obtienen mejores resultados acorde a las metricas de validación cruzada es:

In [ ]:
modelo_final['modelo'].get_params()

In [ ]:
# Diagnóstico errores (residuos) de las predicciones de entrenamiento
# ==============================================================================
#Ytrain = Ytrain.flatten()
#prediccion_train = modelo.predict(exog = Xtrain)
residuos_test   = predicciones - y_test

In [ ]:
import statsmodels.api as sm
# Gráficos
# ==============================================================================
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(8, 14))

axes[0, 0].scatter(y_test, predicciones, edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
                'k--', color = 'black', lw=2)
axes[0, 0].set_title('Valor predicho vs valor real', fontsize = 10, fontweight = "bold")
axes[0, 0].set_xlabel('Real')
axes[0, 0].set_ylabel('Predicción')
axes[0, 0].tick_params(labelsize = 7)

axes[0, 1].scatter(list(range(len(y_test))), residuos_train,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 1].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[0, 1].set_title('Residuos del modelo', fontsize = 10, fontweight = "bold")
axes[0, 1].set_xlabel('id')
axes[0, 1].set_ylabel('Residuo')
axes[0, 1].tick_params(labelsize = 7)

sns.histplot(
    data    = residuos_test,
    stat    = "density",
    kde     = True,
    line_kws= {'linewidth': 1},
    color   = "firebrick",
    alpha   = 0.3,
    ax      = axes[1, 0]
)

axes[1, 0].set_title('Distribución residuos del modelo', fontsize = 10,
                     fontweight = "bold")
axes[1, 0].set_xlabel("Residuo")
axes[1, 0].tick_params(labelsize = 7)


sm.qqplot(
    residuos_test,
    fit   = True,
    line  = 'q',
    ax    = axes[1, 1], 
    color = 'firebrick',
    alpha = 0.4,
    lw    = 2
)
axes[1, 1].set_title('Q-Q residuos del modelo', fontsize = 10, fontweight = "bold")
axes[1, 1].tick_params(labelsize = 7)

axes[2, 0].scatter(predicciones, residuos_test,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[2, 0].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[2, 0].set_title('Residuos del modelo vs predicción', fontsize = 10, fontweight = "bold")
axes[2, 0].set_xlabel('Predicción')
axes[2, 0].set_ylabel('Residuo')
axes[2, 0].tick_params(labelsize = 7)

# Se eliminan los axes vacíos
fig.delaxes(axes[2,1])

fig.tight_layout()
plt.subplots_adjust(top=0.9)
fig.suptitle('Diagnóstico residuos', fontsize = 12, fontweight = "bold");
plt.show()

In [ ]:
from sklearn.metrics import r2_score , mean_absolute_error, mean_squared_error
score = r2_score(y_test,predicciones)
mae = mean_absolute_error(y_test, predicciones)
mse = mean_squared_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
print("Rendimiento en pruebas:")
print("-------------------------------")
print('r2_score is', score)
print('MAE is ', mae)
print('MSE is ', mse)
print('RMSE is ',rmse)

In [ ]:
y_test

In [ ]:
init=5
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(12, 9))
real=y_test.to_numpy()[init:init+30]
plt.plot(real,marker='o',alpha=0.6, label="valor real")
pred=predicciones#
p=pred[init:init+30]
plt.plot(p,marker='*',alpha=0.6, label="valor predicción")
axes.set_title("Valor real vs prediccion", fontsize = 10, fontweight = "bold")
axes.tick_params(labelsize = 8)
axes.set_xlabel("# muestra")
axes.set_ylabel("Precio de venta")
legend = axes.legend(loc='upper center', shadow=True, fontsize='x-large')
fig.tight_layout()
plt.show()

KERAS

In [ ]:
X_train_prep.shape

In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import set_random_seed
import tensorflow as tf
np.random.seed(25)
set_random_seed(812)

# This will make TensorFlow ops as deterministic as possible, but it will
# affect the overall performance, so it's not enabled by default.
# `enable_op_determinism()` is introduced in TensorFlow 2.9.
tf.config.experimental.enable_op_determinism()

In [ ]:
model = Sequential()
model = Sequential([
    Dense(16, activation='tanh', input_shape=[17]),#len(train_dataset.keys())
    Dense(16, activation='relu'),
    Dense(12, activation='relu'),
    Dense(1)
  ])
#relu64-relu64 59%
 
model.compile(loss='mean_squared_error',
              optimizer='adam',
              metrics=['mae', 'mse'])#['MeanSquaredError'])
 
model.fit(X_train_prep, y_train, epochs=800)
 
# evaluamos el modelo
scores = model.evaluate(X_test_prep, y_test)
 
#print("\n%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
#print (model.predict(X_test_prep).round())

In [ ]:
model.summary()

In [ ]:
predictions=model.predict(X_test_prep).flatten()

In [ ]:
predictions[0:10]

In [ ]:
from sklearn.metrics import r2_score , mean_absolute_error, mean_squared_error
score = r2_score(y_test,predictions)
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test,predictions))
print("Rendimiento en pruebas:")
print("-------------------------------")
print('r2_score is', score)
print('MAE is ', mae)
print('MSE is ', mse)
print('RMSE is ',rmse)

In [ ]:
predicciones=predictions
# Gráficos
# ==============================================================================
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(8, 14))

axes[0, 0].scatter(y_test, predicciones, edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
                'k--', color = 'black', lw=2)
axes[0, 0].set_title('Valor predicho vs valor real', fontsize = 10, fontweight = "bold")
axes[0, 0].set_xlabel('Real')
axes[0, 0].set_ylabel('Predicción')
axes[0, 0].tick_params(labelsize = 7)

axes[0, 1].scatter(list(range(len(y_test))), residuos_train,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[0, 1].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[0, 1].set_title('Residuos del modelo', fontsize = 10, fontweight = "bold")
axes[0, 1].set_xlabel('id')
axes[0, 1].set_ylabel('Residuo')
axes[0, 1].tick_params(labelsize = 7)

sns.histplot(
    data    = residuos_test,
    stat    = "density",
    kde     = True,
    line_kws= {'linewidth': 1},
    color   = "firebrick",
    alpha   = 0.3,
    ax      = axes[1, 0]
)

axes[1, 0].set_title('Distribución residuos del modelo', fontsize = 10,
                     fontweight = "bold")
axes[1, 0].set_xlabel("Residuo")
axes[1, 0].tick_params(labelsize = 7)


sm.qqplot(
    residuos_test,
    fit   = True,
    line  = 'q',
    ax    = axes[1, 1], 
    color = 'firebrick',
    alpha = 0.4,
    lw    = 2
)
axes[1, 1].set_title('Q-Q residuos del modelo', fontsize = 10, fontweight = "bold")
axes[1, 1].tick_params(labelsize = 7)

axes[2, 0].scatter(predicciones, residuos_test,
                   edgecolors=(0, 0, 0), alpha = 0.4)
axes[2, 0].axhline(y = 0, linestyle = '--', color = 'black', lw=2)
axes[2, 0].set_title('Residuos del modelo vs predicción', fontsize = 10, fontweight = "bold")
axes[2, 0].set_xlabel('Predicción')
axes[2, 0].set_ylabel('Residuo')
axes[2, 0].tick_params(labelsize = 7)

# Se eliminan los axes vacíos
fig.delaxes(axes[2,1])

fig.tight_layout()
plt.subplots_adjust(top=0.9)
fig.suptitle('Diagnóstico residuos', fontsize = 12, fontweight = "bold");
plt.show()

In [ ]:
import numpy as np
from viznet import connecta2a, node_sequence, NodeBrush, EdgeBrush, DynamicShow


def draw_feed_forward(ax, num_node_list):
    '''
    draw a feed forward neural network.

    Args:
        num_node_list (list<int>): number of nodes in each layer.
    '''
    num_hidden_layer = len(num_node_list) - 2
    token_list = ['\sigma^z'] + \
        ['y^{(%s)}' % (i + 1) for i in range(num_hidden_layer)] + ['\psi']
    kind_list = ['nn.input'] + ['nn.hidden'] * num_hidden_layer + ['nn.output']
    radius_list = [0.3] + [0.2] * num_hidden_layer + [0.3]
    y_list = 3.5 * np.arange(len(num_node_list))

    seq_list = []
    for n, kind, radius, y in zip(num_node_list, kind_list, radius_list, y_list):
        b = NodeBrush(kind, ax)
        seq_list.append(node_sequence(b, n, center=(0, y)))

    eb = EdgeBrush('-->', ax)
    for st, et in zip(seq_list[:-1], seq_list[1:]):
        connecta2a(st, et, eb)


def real_bp():
    with DynamicShow((30, 40), '_feed_forward.png') as d:
        draw_feed_forward(d.ax, num_node_list=[17,16,12,12, 1])


if __name__ == '__main__':
    real_bp()